# Conjuntos de datos

In [109]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Raíz del proyecto y carpetas de datos

Comprobamos que `ROOT` apunta a la raíz (allí está `pyproject.toml`) y que las carpetas de datos existen. Como la ruta se calcula desde `config.py`, el resultado no depende de la carpeta desde donde se ejecute el notebook.

In [110]:
from threat_detector.config import PROCESSED_DIR, RAW_DIR, ROOT

print(f"Carpeta raíz: {ROOT.name}")
print(f"¿Tiene pyproject.toml?: {(ROOT / 'pyproject.toml').exists()}")
for folder in (RAW_DIR, PROCESSED_DIR):
    print(f"{folder.relative_to(ROOT)} existe: {folder.exists()}")
print(f"Archivos en data/raw: {sorted(path.name for path in RAW_DIR.iterdir())}")

Carpeta raíz: network-traffic-threat-detector
¿Tiene pyproject.toml?: True
data/raw existe: True
data/processed existe: True
Archivos en data/raw: ['NUSW-NB15_features.csv', 'UNSW_NB15_testing-set.csv', 'UNSW_NB15_training-set.csv']


## 2. Rutas de los tres CSV

Comprobamos que cada ruta de `config.py` apunta a un archivo que existe y mostramos su tamaño. `data/README.md` indica que los tres pesan cerca de 47 MB en total.

In [111]:
from threat_detector.config import FEATURES_CSV, TEST_CSV, TRAIN_CSV

for path in (TRAIN_CSV, TEST_CSV, FEATURES_CSV):
    print(f"{path.name}: existe = {path.is_file()}, {path.stat().st_size / 1e6:.3f} MB")

UNSW_NB15_training-set.csv: existe = True, 32.293 MB
UNSW_NB15_testing-set.csv: existe = True, 15.381 MB
NUSW-NB15_features.csv: existe = True, 0.004 MB


## 3. Rutas de los artefactos

Mostramos a dónde apunta cada ruta y si existe. `artifacts/models/` y `registry.json` no existen hasta que se exporta un modelo.

In [112]:
from threat_detector.config import ARTIFACTS_DIR, FIGURES_DIR, MODELS_DIR, REGISTRY_PATH

for path in (ARTIFACTS_DIR, MODELS_DIR, FIGURES_DIR, REGISTRY_PATH):
    print(f"{path.relative_to(ROOT)}: existe = {path.exists()}")
print(f"Figuras guardadas: {len(list(FIGURES_DIR.glob('*.png')))}")

artifacts: existe = True
artifacts/models: existe = False
artifacts/figures: existe = True
artifacts/registry.json: existe = False
Figuras guardadas: 14


## 4. Ruta de los archivos estáticos

Comprobamos que `STATIC_DIR` queda dentro del paquete, junto a `config.py`. La carpeta no existe hasta que se crea la app.

In [113]:
from pathlib import Path

from threat_detector import config
from threat_detector.config import STATIC_DIR

print(f"{STATIC_DIR.relative_to(ROOT)}: existe = {STATIC_DIR.exists()}")
print(f"¿Está dentro del paquete?: {STATIC_DIR.is_relative_to(Path(config.__file__).parent)}")

src/threat_detector/app/static: existe = True
¿Está dentro del paquete?: True


## 5. Parámetros de la separación

Mostramos cuántos flujos del archivo de entrenamiento corresponden a validación con `VALIDATION_SIZE`. También comprobamos que `RANDOM_STATE` hace repetible el azar: dos muestras con la misma semilla traen los mismos flujos, y con otra semilla no.

In [114]:
import pandas as pd

from threat_detector.config import RANDOM_STATE, TRAIN_CSV, VALIDATION_SIZE

flow_ids = pd.read_csv(TRAIN_CSV, usecols=["id"])["id"]
validation_flows = round(len(flow_ids) * VALIDATION_SIZE)
print(f"Flujos del archivo de entrenamiento: {len(flow_ids):,}")
print(f"Validación ({VALIDATION_SIZE:.0%}): cerca de {validation_flows:,} flujos")
print(f"Entrenamiento ({1 - VALIDATION_SIZE:.0%}): cerca de {len(flow_ids) - validation_flows:,} flujos")

first = flow_ids.sample(1000, random_state=RANDOM_STATE)
second = flow_ids.sample(1000, random_state=RANDOM_STATE)
other_seed = flow_ids.sample(1000, random_state=RANDOM_STATE + 1)
print(f"¿Misma semilla, mismos flujos?: {first.equals(second)}")
print(f"¿Otra semilla, mismos flujos?: {first.equals(other_seed)}")

Flujos del archivo de entrenamiento: 175,341
Validación (20%): cerca de 35,068 flujos
Entrenamiento (80%): cerca de 140,273 flujos
¿Misma semilla, mismos flujos?: True
¿Otra semilla, mismos flujos?: False


## 6. Parámetros de la agrupación de categorías

Contamos, con el archivo de entrenamiento, cuántas categorías de `proto`, `service` y `state` alcanzan `MIN_CATEGORY_COUNT`. Sumando `OTHER_CATEGORY` a cada variable deben salir 21 columnas one-hot. También comprobamos que ninguna categoría real se llame como `OTHER_CATEGORY`, porque se mezclaría con las agrupadas.

In [115]:
from threat_detector.config import MIN_CATEGORY_COUNT, OTHER_CATEGORY
from threat_detector.dataset import CATEGORICAL_COLS

categories = pd.read_csv(TRAIN_CSV, usecols=CATEGORICAL_COLS)

total_columns = 0
for col in CATEGORICAL_COLS:
    counts = categories[col].value_counts()
    kept = counts.index[counts >= MIN_CATEGORY_COUNT].tolist()
    total_columns += len(kept) + 1
    print(f"{col}: {len(kept)} de {len(counts)} categorías se conservan -> {kept}")
    print(f"  Flujos en '{OTHER_CATEGORY}': {counts[counts < MIN_CATEGORY_COUNT].sum():,}")
    print(f"  ¿Ya existe una categoría '{OTHER_CATEGORY}'?: {OTHER_CATEGORY in counts.index}")
print(f"Columnas one-hot: {total_columns}")

proto: 6 de 133 categorías se conservan -> ['tcp', 'udp', 'unas', 'arp', 'ospf', 'sctp']
  Flujos en 'other': 13,424
  ¿Ya existe una categoría 'other'?: False
service: 8 de 13 categorías se conservan -> ['-', 'dns', 'http', 'smtp', 'ftp-data', 'ftp', 'ssh', 'pop3']
  Flujos en 'other': 267
  ¿Ya existe una categoría 'other'?: False
state: 4 de 9 categorías se conservan -> ['INT', 'FIN', 'CON', 'REQ']
  Flujos en 'other': 98
  ¿Ya existe una categoría 'other'?: False
Columnas one-hot: 21


## 7. Detección mínima para el umbral

Traducimos `TARGET_RECALL` a flujos: con los ataques del archivo de entrenamiento estimamos cuántos caerían en validación y cuántos de ellos deben quedar por encima del umbral. Son cifras aproximadas, porque la separación por grupos no da exactamente el 20 %.

In [116]:
import math

from threat_detector.config import TARGET_RECALL

attacks = int(pd.read_csv(TRAIN_CSV, usecols=["label"])["label"].sum())
validation_attacks = round(attacks * VALIDATION_SIZE)
required_detections = math.ceil(validation_attacks * TARGET_RECALL)

print(f"Ataques en el archivo de entrenamiento: {attacks:,}")
print(f"Ataques en validación: cerca de {validation_attacks:,}")
print(f"Deben quedar por encima del umbral ({TARGET_RECALL:.0%}): cerca de {required_detections:,}")
print(f"Pueden quedar por debajo: cerca de {validation_attacks - required_detections:,}")

Ataques en el archivo de entrenamiento: 119,341
Ataques en validación: cerca de 23,868
Deben quedar por encima del umbral (95%): cerca de 22,675
Pueden quedar por debajo: cerca de 1,193


## 8. Límites de severidad

Mostramos el rango de probabilidad de ataque de cada severidad y comprobamos que los límites van de mayor a menor, entre 0 y 1. Esta prueba no usa los datos, porque la severidad se calcula con la probabilidad que da un modelo.

In [117]:
from threat_detector.config import SEVERITY_CRITICAL, SEVERITY_HIGH, SEVERITY_MEDIUM

print(f"Crítica: {SEVERITY_CRITICAL:.2f} o más")
print(f"Alta:    de {SEVERITY_HIGH:.2f} a menos de {SEVERITY_CRITICAL:.2f}")
print(f"Media:   de {SEVERITY_MEDIUM:.2f} a menos de {SEVERITY_HIGH:.2f}")
print(f"Baja:    del umbral del modelo a menos de {SEVERITY_MEDIUM:.2f}")
print(f"¿Límites en orden?: {1 > SEVERITY_CRITICAL > SEVERITY_HIGH > SEVERITY_MEDIUM > 0}")

Crítica: 0.90 o más
Alta:    de 0.75 a menos de 0.90
Media:   de 0.60 a menos de 0.75
Baja:    del umbral del modelo a menos de 0.60
¿Límites en orden?: True


## 9. Tamaño de la muestra de demostración

Comprobamos que la muestra se puede armar con estos valores: cada tipo de ataque del conjunto de prueba debe tener al menos `DEMO_MIN_PER_ATTACK` flujos, y los mínimos juntos deben caber en `DEMO_SAMPLE_SIZE`. De la prueba solo se cuentan los flujos por tipo.

In [118]:
from threat_detector.config import DEMO_MIN_PER_ATTACK, DEMO_SAMPLE_SIZE, TEST_CSV

test_counts = pd.read_csv(TEST_CSV, usecols=["attack_cat"])["attack_cat"].value_counts()
attack_counts = test_counts.drop("Normal")
reserved = len(attack_counts) * DEMO_MIN_PER_ATTACK

print(attack_counts.to_string())
print(f"\nTipos de ataque: {len(attack_counts)}")
print(f"¿Todos tienen al menos {DEMO_MIN_PER_ATTACK} flujos?: {(attack_counts >= DEMO_MIN_PER_ATTACK).all()}")
print(f"Flujos reservados por los mínimos: {reserved} de {DEMO_SAMPLE_SIZE}")
print(f"Flujos restantes, al azar: {DEMO_SAMPLE_SIZE - reserved}")

attack_cat
Generic           18871
Exploits          11132
Fuzzers            6062
DoS                4089
Reconnaissance     3496
Analysis            677
Backdoor            583
Shellcode           378
Worms                44

Tipos de ataque: 9
¿Todos tienen al menos 10 flujos?: True
Flujos reservados por los mínimos: 90 de 500
Flujos restantes, al azar: 410


## 10. Columnas de identificador y etiquetas

Comprobamos en el archivo de entrenamiento que `ID_COL` no se repite, que `TARGET` solo vale 0 o 1 y que `ATTACK_CAT` tiene `Normal` más 9 tipos de ataque. También comprobamos que las dos etiquetas coinciden: un flujo es normal en una si y solo si lo es en la otra.

In [119]:
from threat_detector.dataset import ATTACK_CAT, ID_COL, TARGET

columns = pd.read_csv(TRAIN_CSV, usecols=[ID_COL, TARGET, ATTACK_CAT])

print(f"¿{ID_COL} sin repetidos?: {columns[ID_COL].is_unique}")
print(f"Valores de {TARGET}: {sorted(columns[TARGET].unique().tolist())}")
print(f"Valores de {ATTACK_CAT} ({columns[ATTACK_CAT].nunique()}): {sorted(columns[ATTACK_CAT].unique().tolist())}")
print(f"¿Las dos etiquetas coinciden?: {((columns[TARGET] == 0) == (columns[ATTACK_CAT] == 'Normal')).all()}")

¿id sin repetidos?: True
Valores de label: [0, 1]
Valores de attack_cat (10): ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Normal', 'Reconnaissance', 'Shellcode', 'Worms']
¿Las dos etiquetas coinciden?: True


## 11. Etiquetas y variables categóricas

Comprobamos en el archivo de entrenamiento que las únicas columnas de texto son `CATEGORICAL_COLS` y `ATTACK_CAT`, y que sacando `ID_COL`, `TARGET_COLS` y `CATEGORICAL_COLS` todas las demás columnas son numéricas. Esas son las variables numéricas de partida, antes de aplicar las decisiones de preparación.

In [120]:
from threat_detector.dataset import CATEGORICAL_COLS, TARGET_COLS

train_columns = pd.read_csv(TRAIN_CSV)
text_cols = train_columns.select_dtypes(exclude="number").columns.tolist()
other_cols = [c for c in train_columns.columns if c not in [ID_COL, *TARGET_COLS, *CATEGORICAL_COLS]]

print(f"TARGET_COLS: {TARGET_COLS}")
print(f"CATEGORICAL_COLS: {CATEGORICAL_COLS}")
print(f"Columnas de texto: {text_cols}")
print(f"¿Son exactamente las categóricas y attack_cat?: {set(text_cols) == {*CATEGORICAL_COLS, ATTACK_CAT}}")
print(f"Columnas restantes: {len(other_cols)}; ¿todas numéricas?: {train_columns[other_cols].apply(pd.api.types.is_numeric_dtype).all()}")

TARGET_COLS: ['label', 'attack_cat']
CATEGORICAL_COLS: ['proto', 'service', 'state']
Columnas de texto: ['proto', 'service', 'state', 'attack_cat']
¿Son exactamente las categóricas y attack_cat?: True
Columnas restantes: 39; ¿todas numéricas?: True


## 12. Columnas de las particiones

Comparamos `RAW_COLUMNS` con el encabezado de los dos archivos: deben tener las mismas columnas y en el mismo orden. Con `nrows=0` solo se lee el encabezado, sin cargar flujos. También comprobamos que incluye las columnas de identificador, etiquetas y categóricas definidas antes.

In [121]:
from threat_detector.dataset import RAW_COLUMNS

print(f"Columnas en RAW_COLUMNS: {len(RAW_COLUMNS)}, sin repetir: {len(set(RAW_COLUMNS))}")
for path in (TRAIN_CSV, TEST_CSV):
    header = pd.read_csv(path, nrows=0).columns.tolist()
    print(f"{path.name}: ¿mismas columnas y mismo orden?: {header == RAW_COLUMNS}")
named_cols = {ID_COL, *TARGET_COLS, *CATEGORICAL_COLS}
print(f"¿Incluye las {len(named_cols)} columnas definidas antes?: {named_cols <= set(RAW_COLUMNS)}")

Columnas en RAW_COLUMNS: 45, sin repetir: 45
UNSW_NB15_training-set.csv: ¿mismas columnas y mismo orden?: True
UNSW_NB15_testing-set.csv: ¿mismas columnas y mismo orden?: True
¿Incluye las 6 columnas definidas antes?: True


## 13. Número de filas

Comparamos `TRAIN_ROWS` y `TEST_ROWS` con las filas reales de cada archivo. Se lee una sola columna, porque solo hace falta contar.

In [122]:
from threat_detector.dataset import TEST_ROWS, TRAIN_ROWS

for path, expected in ((TRAIN_CSV, TRAIN_ROWS), (TEST_CSV, TEST_ROWS)):
    rows = len(pd.read_csv(path, usecols=[ID_COL]))
    print(f"{path.name}: {rows:,} filas, esperadas {expected:,} -> ¿coinciden?: {rows == expected}")

UNSW_NB15_training-set.csv: 175,341 filas, esperadas 175,341 -> ¿coinciden?: True
UNSW_NB15_testing-set.csv: 82,332 filas, esperadas 82,332 -> ¿coinciden?: True


## 14. Nombres de la descripción y de las particiones

Aplicamos `DESCRIPTION_RENAMES` a los nombres de `NUSW-NB15_features.csv`, pasados a minúsculas, y comparamos con `RAW_COLUMNS`. Deben quedar solo las diferencias conocidas: 6 variables que las particiones no traen y 2 columnas que solo existen en ellas.

In [123]:
from threat_detector.config import FEATURES_CSV
from threat_detector.dataset import DESCRIPTION_RENAMES

described_names = pd.read_csv(FEATURES_CSV, encoding="cp1252")["Name"].str.strip().str.lower()
print(f"¿Todos los cambios de nombre se usan?: {set(DESCRIPTION_RENAMES) <= set(described_names)}")

renamed = set(described_names.replace(DESCRIPTION_RENAMES))
print(f"Variables en la descripción: {len(renamed)}")
print(f"Faltan en las particiones: {sorted(renamed - set(RAW_COLUMNS))}")
print(f"Solo en las particiones: {sorted(set(RAW_COLUMNS) - renamed)}")

¿Todos los cambios de nombre se usan?: True
Variables en la descripción: 49
Faltan en las particiones: ['dsport', 'dstip', 'ltime', 'sport', 'srcip', 'stime']
Solo en las particiones: ['id', 'rate']


## 15. Carga y verificación de un archivo de flujos

Primero cargamos el archivo de entrenamiento real con `load_flows`. Después comprobamos sus tres errores con archivos temporales: uno que no existe, uno al que le falta una columna y uno con menos filas de las esperadas. Los temporales se borran al terminar la celda.

In [124]:
import tempfile
from pathlib import Path

from threat_detector.dataset import TARGET, load_flows

flows = load_flows(TRAIN_CSV, TRAIN_ROWS)
print(f"Archivo correcto: {flows.shape[0]:,} flujos x {flows.shape[1]} columnas\n")


def show_error(path, expected_rows):
    """Muestra el error que lanza load_flows, para ver el mensaje que recibiría el equipo."""
    try:
        load_flows(path, expected_rows)
        print("Sin error: la verificación no detectó el problema")
    except (FileNotFoundError, ValueError) as error:
        print(f"{type(error).__name__}: {error}\n")


with tempfile.TemporaryDirectory() as temp_dir:
    temp_dir = Path(temp_dir)

    show_error(temp_dir / "archivo_inexistente.csv", TRAIN_ROWS)

    no_label = temp_dir / "sin_etiqueta.csv"
    flows.head(5).drop(columns=TARGET).to_csv(no_label, index=False)
    show_error(no_label, 5)

    incomplete = temp_dir / "incompleto.csv"
    flows.head(5).to_csv(incomplete, index=False)
    show_error(incomplete, TRAIN_ROWS)

Archivo correcto: 175,341 flujos x 45 columnas

FileNotFoundError: No se encontró el archivo archivo_inexistente.csv. Revisa data/README.md para saber cómo descargarlo.

ValueError: sin_etiqueta.csv no tiene las columnas esperadas. Faltan: ['label']. Sobran: [].

ValueError: incompleto.csv tiene 5 filas y se esperaban 175,341. La descarga puede estar incompleta o ser otra versión del dataset.



## 16. Carga de las dos particiones

Cargamos cada partición y comparamos sus flujos normales y de ataque con la tabla de `data/README.md`: 56.000 y 119.341 en entrenamiento; 37.000 y 45.332 en prueba. De la prueba solo se cuentan flujos.

In [125]:
from threat_detector.dataset import load_test, load_train

for name, load in (("Entrenamiento", load_train), ("Prueba", load_test)):
    partition = load()
    normal, attack = partition[TARGET].value_counts().sort_index()
    print(f"{name}: {partition.shape[0]:,} flujos x {partition.shape[1]} columnas; "
          f"normal = {normal:,}, ataque = {attack:,}")

Entrenamiento: 175,341 flujos x 45 columnas; normal = 56,000, ataque = 119,341
Prueba: 82,332 flujos x 45 columnas; normal = 37,000, ataque = 45,332


## 17. Descripción de las variables

Cargamos la descripción y comprobamos cuatro cosas: que tiene las 49 variables, que sus nombres coinciden con `RAW_COLUMNS` salvo las diferencias conocidas, que el apóstrofo se leyó bien y que los nombres de las columnas quedaron sin espacios.

In [126]:
from threat_detector.dataset import load_feature_descriptions

descriptions = load_feature_descriptions()
described = set(descriptions["name"])
text = descriptions["description"].str.cat(sep=" ")

print(f"Variables: {len(descriptions)}; columnas: {descriptions.columns.tolist()}")
print(f"Faltan en las particiones: {sorted(described - set(RAW_COLUMNS))}")
print(f"Solo en las particiones: {sorted(set(RAW_COLUMNS) - described)}")
print(f"¿Apóstrofo bien leído?: {'’' in text}; ¿queda el carácter invisible?: {chr(0x92) in text}")

descriptions[descriptions["name"].isin(["smean", "ct_src_ltm", "sttl"])]

Variables: 49; columnas: ['no.', 'name', 'type', 'description']
Faltan en las particiones: ['dsport', 'dstip', 'ltime', 'sport', 'srcip', 'stime']
Solo en las particiones: ['id', 'rate']
¿Apóstrofo bien leído?: True; ¿queda el carácter invisible?: False


,no.,name,type,description
9,10,sttl,Integer,Source to destination time to live value
22,23,smean,integer,Mean of the ?ow packet size transmitted by the...
43,44,ct_src_ltm,integer,No. of connections of the same source address ...


## 18. Variables de entrada

`INPUT_COLS` son las columnas que describen el flujo: todas menos `id`, `label` y `attack_cat`. Con ellas se decide qué flujos son idénticos al separar entrenamiento y validación.

Comprobamos que son 42, que quedaron fuera justo el identificador y las dos etiquetas, que conservan el orden del archivo y que, fuera de las 3 categóricas, todas son numéricas.

In [127]:
from threat_detector.dataset import INPUT_COLS

train_flows = load_train()
numeric_inputs = [c for c in INPUT_COLS if c not in CATEGORICAL_COLS]

print(f"Variables de entrada: {len(INPUT_COLS)} ({len(CATEGORICAL_COLS)} categóricas y {len(numeric_inputs)} numéricas)")
print(f"Quedaron fuera: {[c for c in RAW_COLUMNS if c not in INPUT_COLS]}")
print(f"¿Conservan el orden del archivo?: {INPUT_COLS == [c for c in train_flows.columns if c in INPUT_COLS]}")
print(f"¿Las no categóricas son todas numéricas?: {all(pd.api.types.is_numeric_dtype(train_flows[c]) for c in numeric_inputs)}")

Variables de entrada: 42 (3 categóricas y 39 numéricas)
Quedaron fuera: ['id', 'attack_cat', 'label']
¿Conservan el orden del archivo?: True
¿Las no categóricas son todas numéricas?: True


## 19. Patrón de cada flujo

`flow_pattern_ids` le asigna a cada flujo un número que resume sus valores en `INPUT_COLS`. Los flujos idénticos comparten el número y forman un grupo; al separar entrenamiento y validación, cada grupo va completo a una de las dos partes.

Comprobamos que da las mismas cifras del análisis exploratorio (101.040 patrones, grupo más grande de 415 flujos, 127 patrones de `Worms`), que el número no cambia si se desordenan las filas y que las etiquetas no entran en él: hay patrones con flujos normales y de ataque a la vez.

In [128]:
from threat_detector.dataset import ATTACK_CAT
from threat_detector.ml.preprocessing import flow_pattern_ids

pattern_ids = flow_pattern_ids(train_flows)
repeated = pattern_ids.duplicated()

print(f"Flujos: {len(pattern_ids):,}; patrones distintos: {pattern_ids.nunique():,}")
print(f"Flujos que repiten un patrón: {repeated.sum():,} ({repeated.mean():.1%})")
print(f"Grupo más grande: {pattern_ids.value_counts().max()} flujos")
print(f"Patrones distintos de Worms: {pattern_ids[train_flows[ATTACK_CAT] == 'Worms'].nunique()}")

shuffled = train_flows.sample(frac=1, random_state=RANDOM_STATE)
print(f"¿Mismo número con las filas desordenadas?: {flow_pattern_ids(shuffled).sort_index().equals(pattern_ids)}")
print(f"Patrones con flujos normales y de ataque: {train_flows.groupby(pattern_ids)[TARGET].nunique().gt(1).sum():,}")

Flujos: 175,341; patrones distintos: 101,040
Flujos que repiten un patrón: 74,301 (42.4%)
Grupo más grande: 415 flujos
Patrones distintos de Worms: 127
¿Mismo número con las filas desordenadas?: True
Patrones con flujos normales y de ataque: 229


## 20. Separación de entrenamiento y validación

`split_train_validation` separa el archivo de entrenamiento en dos partes: una para entrenar y otra para validar. La separación se hace por grupos de flujos idénticos y se estratifica por `attack_cat`. No se elimina ningún flujo antes de separar.

Comprobamos cuatro cosas:
- La validación tiene el 20 % de los flujos y entre las dos partes suman el archivo completo, sin flujos en ambas.
- Ningún flujo de validación es idéntico a uno de entrenamiento. Lo comparamos con los valores de `INPUT_COLS`, sin usar el patrón.
- La proporción de cada `attack_cat` es la misma en las dos partes (la diferencia se muestra en puntos porcentuales).
- `Worms`, la clase más pequeña, tiene flujos y patrones distintos en la validación.

La separación tarda unos 15 segundos.

In [129]:
from threat_detector.ml.preprocessing import split_train_validation

train_part, validation_part = split_train_validation(train_flows)

print(f"Entrenamiento: {len(train_part):,} flujos; validación: {len(validation_part):,} ({len(validation_part) / len(train_flows):.2%})")
print(f"¿Suman el archivo completo?: {len(train_part) + len(validation_part) == len(train_flows)}")
print(f"¿Algún flujo está en las dos partes?: {train_part.index.isin(validation_part.index).any()}")

shared = train_part[INPUT_COLS].drop_duplicates().merge(validation_part[INPUT_COLS].drop_duplicates())
print(f"Flujos de validación idénticos a uno de entrenamiento: {len(shared)}")

worms = validation_part[validation_part[ATTACK_CAT] == "Worms"]
print(f"Worms en validación: {len(worms)} flujos, {flow_pattern_ids(worms).nunique()} patrones distintos")

proportions = pd.DataFrame({
    "entrenamiento": train_part[ATTACK_CAT].value_counts(normalize=True),
    "validación": validation_part[ATTACK_CAT].value_counts(normalize=True),
})
proportions["diferencia (pp)"] = (proportions["validación"] - proportions["entrenamiento"]) * 100
proportions.round(4)

Entrenamiento: 140,273 flujos; validación: 35,068 (20.00%)
¿Suman el archivo completo?: True
¿Algún flujo está en las dos partes?: False
Flujos de validación idénticos a uno de entrenamiento: 0
Worms en validación: 26 flujos, 26 patrones distintos


,entrenamiento,validación,diferencia (pp)
attack_cat,,,
Normal,0.3194,0.3194,0.0002
Generic,0.2281,0.2281,0.0002
Exploits,0.1904,0.1905,0.0016
Fuzzers,0.1037,0.1037,-0.0028
DoS,0.0699,0.0699,0.0008
Reconnaissance,0.0598,0.0598,-0.0007
Analysis,0.0114,0.0114,0.0000
Backdoor,0.0100,0.0100,-0.0007
Shellcode,0.0065,0.0065,0.0014


## 21. La separación siempre da lo mismo

Como la separación usa la semilla fija `RANDOM_STATE`, no se guarda en disco: se vuelve a calcular cada vez. Para que eso sea válido, debe dar exactamente el mismo resultado en cada ejecución. La corremos otra vez y comparamos las dos partes, fila por fila y columna por columna.

In [130]:
train_again, validation_again = split_train_validation(train_flows)

print(f"¿Mismo entrenamiento?: {train_again.equals(train_part) and train_again.index.equals(train_part.index)}")
print(f"¿Misma validación?: {validation_again.equals(validation_part) and validation_again.index.equals(validation_part.index)}")

¿Mismo entrenamiento?: True
¿Misma validación?: True


## 22. Ruta de la muestra de demostración

`DEMO_SAMPLE_CSV` es la ruta de `sample_flows.csv`, la muestra que usa la app. Va dentro de los archivos estáticos para que la app la pueda ofrecer sin depender de `data/`, que no se sube a git.

Comprobamos que queda dentro de `STATIC_DIR`, que esa carpeta existe y si el archivo ya fue generado.

In [131]:
from threat_detector.config import DEMO_SAMPLE_CSV, STATIC_DIR

print(f"Ruta: {DEMO_SAMPLE_CSV.relative_to(ROOT)}")
print(f"¿Está dentro de STATIC_DIR?: {DEMO_SAMPLE_CSV.parent == STATIC_DIR}")
print(f"¿Existe la carpeta?: {STATIC_DIR.is_dir()}")
print(f"¿Ya se generó el archivo?: {DEMO_SAMPLE_CSV.is_file()}")

Ruta: src/threat_detector/app/static/sample_flows.csv
¿Está dentro de STATIC_DIR?: True
¿Existe la carpeta?: True
¿Ya se generó el archivo?: True


## 23. Selección de la muestra de demostración

`make_demo_sample` elige los 500 flujos de la muestra a partir del conjunto de prueba y los devuelve sin guardar nada. Primero toma 10 flujos de cada tipo de ataque y luego completa los 500 con flujos al azar, repartidos en la misma proporción de `attack_cat` que la prueba.

Comprobamos que:
- Tiene 500 flujos, al menos 10 de cada uno de los 9 tipos de ataque, y las 45 columnas originales en su orden, incluidas `label` y `attack_cat`.
- Cada flujo es una fila de la prueba sin cambios y ninguno se repite.
- Las filas quedaron mezcladas (no vienen agrupadas por clase).
- Sale exactamente igual si se selecciona dos veces.

La tabla compara cuántos flujos tiene cada clase en la muestra con la proporción de la prueba. `Normal` queda por debajo de su proporción porque los 90 flujos del mínimo son todos ataques.

In [132]:
from threat_detector.config import DEMO_MIN_PER_ATTACK, DEMO_SAMPLE_SIZE
from threat_detector.dataset import ID_COL, make_demo_sample

test_flows = load_test()
sample = make_demo_sample(test_flows)
attack_counts = sample.loc[sample[TARGET] == 1, ATTACK_CAT].value_counts()

print(f"Flujos: {len(sample)} (se esperaban {DEMO_SAMPLE_SIZE})")
print(f"Tipos de ataque: {len(attack_counts)}; el que menos tiene: {attack_counts.idxmin()} con {attack_counts.min()} (mínimo {DEMO_MIN_PER_ATTACK})")
print(f"¿Tiene las 45 columnas en su orden?: {sample.columns.tolist() == RAW_COLUMNS}")

original_rows = test_flows.set_index(ID_COL).loc[sample[ID_COL]].reset_index()[RAW_COLUMNS]
print(f"¿Cada flujo es una fila de la prueba sin cambios?: {original_rows.equals(sample)}")
print(f"¿Algún flujo repetido?: {sample[ID_COL].duplicated().any()}")
print(f"Clases de los primeros 10 flujos: {sample[ATTACK_CAT].head(10).tolist()}")
print(f"¿Sale igual si se selecciona otra vez?: {make_demo_sample(test_flows).equals(sample)}")

composition = pd.DataFrame({
    "flujos en la muestra": sample[ATTACK_CAT].value_counts(),
    "% en la muestra": sample[ATTACK_CAT].value_counts(normalize=True) * 100,
    "% en la prueba": test_flows[ATTACK_CAT].value_counts(normalize=True) * 100,
})
composition.round(1)

Flujos: 500 (se esperaban 500)
Tipos de ataque: 9; el que menos tiene: Worms con 10 (mínimo 10)
¿Tiene las 45 columnas en su orden?: True
¿Cada flujo es una fila de la prueba sin cambios?: True
¿Algún flujo repetido?: False
Clases de los primeros 10 flujos: ['Generic', 'Shellcode', 'Exploits', 'Normal', 'Normal', 'Exploits', 'Exploits', 'Normal', 'Reconnaissance', 'Fuzzers']
¿Sale igual si se selecciona otra vez?: True


,flujos en la muestra,% en la muestra,% en la prueba
attack_cat,,,
Normal,184,36.8,44.9
Generic,104,20.8,22.9
Exploits,66,13.2,13.5
Fuzzers,40,8.0,7.4
DoS,30,6.0,5.0
Reconnaissance,28,5.6,4.2
Analysis,13,2.6,0.8
Backdoor,13,2.6,0.7
Shellcode,12,2.4,0.5


## 24. Archivo de la muestra de demostración

`sample_flows.csv` se genera desde la terminal con `write_demo_sample()` (el comando está en `data/README.md`); este notebook no lo escribe, solo lo verifica.

Lo leemos con `load_flows`, la misma verificación de columnas y filas que se aplica a un archivo que suba el usuario: si le faltara o le sobrara alguna columna, o no tuviera 500 flujos, la celda daría error. Además comprobamos que tiene al menos 10 flujos de cada tipo de ataque, que conserva `label` y `attack_cat`, y que coincide con lo que selecciona hoy `make_demo_sample` (si no coincide, hay que volver a generarlo).

In [133]:
from threat_detector.config import DEMO_SAMPLE_CSV
from threat_detector.dataset import load_flows

saved_sample = load_flows(DEMO_SAMPLE_CSV, DEMO_SAMPLE_SIZE)
saved_attacks = saved_sample.loc[saved_sample[TARGET] == 1, ATTACK_CAT].value_counts()

print(f"Flujos: {len(saved_sample)}; columnas: {saved_sample.shape[1]} (validadas con load_flows)")
print(f"Tipos de ataque: {len(saved_attacks)}; el que menos tiene: {saved_attacks.idxmin()} con {saved_attacks.min()} (mínimo {DEMO_MIN_PER_ATTACK})")
print(f"¿Tiene label y attack_cat?: {set(TARGET_COLS) <= set(saved_sample.columns)}")
print(f"¿Coincide con la selección de make_demo_sample?: {saved_sample.equals(make_demo_sample(test_flows))}")
print(f"Tamaño del archivo: {DEMO_SAMPLE_CSV.stat().st_size / 1024:.0f} KB")

Flujos: 500; columnas: 45 (validadas con load_flows)
Tipos de ataque: 9; el que menos tiene: Worms con 10 (mínimo 10)
¿Tiene label y attack_cat?: True
¿Coincide con la selección de make_demo_sample?: True
Tamaño del archivo: 94 KB
